# 01 · 斷詞 Tokenization

歡迎來到 **大型語言模型 → 從零打造迷你 GPT**。

這個軌道不教你「呼叫 ChatGPT」，而是帶你**親手把一個 GPT 從零刻出來**——從斷詞、注意力、Transformer，一路到訓練、生成、對齊。模型刻意做得很小、功能很爛，**重點是徹底搞懂裡面到底發生什麼事**。

第一步：電腦看不懂文字，只看得懂數字。**Tokenization（斷詞）** 就是把文字切成一個個 token、再對應成數字的過程。它是所有 LLM 的入口。

> 💡 本軌道建議先學完 `ml/pytorch`（需要 PyTorch 基礎）。每格按 ▶️ 執行，建議在 Colab 開（後面訓練可開免費 GPU）。

## 學習目標

- 理解 token 是 LLM 的「原子」
- 實作最簡單的**字元級**斷詞：`encode` / `decode`
- understand **BPE（子詞斷詞）** 的核心想法——真實 LLM 用的方法

## 1. 字元級斷詞：最簡單的做法

最直接的斷詞：**一個字一個 token**。我們用一小段唐詩當語料，把所有出現過的字收集成「詞表」，每個字配一個編號。

In [1]:
text = """床前明月光，疑是地上霜。舉頭望明月，低頭思故鄉。
春眠不覺曉，處處聞啼鳥。夜來風雨聲，花落知多少。
白日依山盡，黃河入海流。欲窮千里目，更上一層樓。
紅豆生南國，春來發幾枝。願君多采擷，此物最相思。
空山不見人，但聞人語響。返景入深林，復照青苔上。
千山鳥飛絕，萬徑人蹤滅。孤舟蓑笠翁，獨釣寒江雪。
松下問童子，言師采藥去。只在此山中，雲深不知處。
人閒桂花落，夜靜春山空。月出驚山鳥，時鳴春澗中。
君自故鄉來，應知故鄉事。來日綺窗前，寒梅著花未。
獨坐幽篁裡，彈琴復長嘯。深林人不知，明月來相照。
山中相送罷，日暮掩柴扉。春草明年綠，王孫歸不歸。
功蓋三分國，名成八陣圖。江流石不轉，遺恨失吞吳。
前不見古人，後不見來者。念天地之悠悠，獨愴然而涕下。
葡萄美酒夜光杯，欲飲琵琶馬上催。醉臥沙場君莫笑，古來征戰幾人回。
秦時明月漢時關，萬里長征人未還。但使龍城飛將在，不教胡馬度陰山。
朝辭白帝彩雲間，千里江陵一日還。兩岸猿聲啼不住，輕舟已過萬重山。
故人西辭黃鶴樓，煙花三月下揚州。孤帆遠影碧空盡，唯見長江天際流。
月落烏啼霜滿天，江楓漁火對愁眠。姑蘇城外寒山寺，夜半鐘聲到客船。
獨在異鄉為異客，每逢佳節倍思親。遙知兄弟登高處，遍插茱萸少一人。
日照香爐生紫煙，遙看瀑布掛前川。飛流直下三千尺，疑是銀河落九天。
國破山河在，城春草木深。感時花濺淚，恨別鳥驚心。
岐王宅裡尋常見，崔九堂前幾度聞。正是江南好風景，落花時節又逢君。
渭城朝雨浥輕塵，客舍青青柳色新。勸君更盡一杯酒，西出陽關無故人。
清明時節雨紛紛，路上行人欲斷魂。借問酒家何處有，牧童遙指杏花村。
千里黃雲白日曛，北風吹雁雪紛紛。莫愁前路無知己，天下誰人不識君。
"""
import torch

# 字元級詞表：每個不同的字 = 一個 token
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
print(f"語料長度 {len(text)} 字，詞表大小 {vocab_size}")

語料長度 715 字，詞表大小 328


In [2]:
# 試試編碼與解碼
s = "明月幾時有"
ids = encode(s)
print("原文:", s)
print("編碼:", ids)
print("解碼:", decode(ids), " ← 還原回原文")

原文: 明月幾時有
編碼: [132, 142, 98, 135, 143]
解碼: 明月幾時有  ← 還原回原文


`encode` 把字串變成數字串、`decode` 變回來。整段語料也被編成一條長長的數字序列 `data`，這就是要餵給模型的東西。

## 2. 真實 LLM 用的是 BPE（子詞斷詞）

字元級簡單，但序列會很長、也學不到「詞」的概念。真實的 GPT 用 **BPE（Byte Pair Encoding）**：從字元開始，**反覆把最常一起出現的相鄰一對合併**成一個新 token，直到詞表夠大。常見的詞變成單一 token，罕見的詞才拆成小塊。

用一個小例子看 BPE 怎麼合併：

In [3]:
from collections import Counter

# 把語料當成 token 序列（先以字元為起點），反覆合併最常見的相鄰對
tokens = list(text.replace("\n", ""))
for step in range(3):
    pairs = Counter(zip(tokens, tokens[1:]))
    best, freq = pairs.most_common(1)[0]
    merged = "".join(best)
    # 合併所有 best 這一對
    new, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == best:
            new.append(merged); i += 2
        else:
            new.append(tokens[i]); i += 1
    tokens = new
    print(f"第 {step + 1} 次合併：把出現 {freq} 次的「{merged}」併成一個 token")
print("\n合併後序列開頭:", tokens[:12])

第 1 次合併：把出現 4 次的「明月」併成一個 token
第 2 次合併：把出現 3 次的「故鄉」併成一個 token
第 3 次合併：把出現 3 次的「千里」併成一個 token

合併後序列開頭: ['床', '前', '明月', '光', '，', '疑', '是', '地', '上', '霜', '。', '舉']


看到了嗎——高頻的字對（像詩裡反覆出現的詞）被併成單一 token。真實的 GPT-2/3/4 就是用這套（在位元組層級）跑幾萬次合併，得到 5 萬 ~ 10 萬大小的詞表。

> 本軌道為了簡單，後面一律用**字元級**斷詞——概念一樣，少了 BPE 的工程細節。

## 小結

- Token 是 LLM 的原子；斷詞把文字 ↔ 數字互轉。
- **字元級**：一字一 token，簡單但序列長。
- **BPE**：反覆合併高頻相鄰對，讓常見詞成為單一 token——真實 LLM 的做法。

## 練習

1. 把 `s` 換成語料裡沒有的字（如「貓」），`encode` 會發生什麼事？想想真實 LLM 怎麼處理未知字（提示：BPE 拆成更小單位）。
2. 把 BPE 的合併次數從 3 改成 10，看會併出哪些 token。

下一課，用這些 token 做 LLM 最核心的任務：**預測下一個字**。